<h1>Prompt Evaluation </h1>

 - Measures prompt effectivenes by evaluating the LLM response on accuracy, relevancy etc
 - Unit testing but for Natural language instructions

There are two types of evaluation:

**Manual Evaluation**  A human scores the output 

**Automated Evaluation** An LLM or algorithm scores the output 

<h2>Why we need Prompt Evaluation </h2>

 - LLMs are non-deterministic and sensitive to wording. A small change in your prompt can **dramatically** change output quality for better or worse.

Prompt evaluation lets you:
1. **Compare** prompt versions objectively
2. **Catch regressions** when models or prompts change
3. **Justify decisions** with data instead of opinion
4. **Iterate faster** by knowing exactly what's improving

## 📊 Metrics for Prompt Effectiveness

We'll focus on **5 core metrics**

| Metric | What it measures | Score Range |
|--------|-----------------|-------------|
| **Accuracy** | Is the information factually correct? | 0.0 – 1.0 |
| **Relevance** | Does the response actually answer the question? | 0.0 – 1.0 |
| **Completeness** | Are all important aspects covered? | 0.0 – 1.0 |
| **Conciseness** | Is the response appropriately brief without losing value? | 0.0 – 1.0 |
| **Clarity** | Is the response easy to understand? | 0.0 – 1.0 |

An **overall score** is computed as the weighted average of all five metrics.


---
## Section 1: Setup


In [4]:
import json
import ollama
import textwrap
import time
from typing import List
import pandas as pd

---
## Section 2: Define Our Test Cases

We'll evaluate prompts on a single real-world task: **Responding with a mail to a customer support query**.


In [17]:
# ── Our test question ──────────────────────────────────────────────────────
QUESTION = """
    A customer purchased a laptop 8 months ago for €1200.
    The laptop battery has started swelling and overheating.
    
    The customer is angry because:
    - Warranty technically expired last week
    - Customer support previously ignored 2 emails
    - They need the laptop urgently for work
    
    Write a customer support response email.
"""

# Key elements a high-quality response should include
REFERENCE_POINTS = [
    "Acknowledges the customer's frustration empathetically",
    "Clearly addresses the battery safety concern",
    "Offers a concrete next step or resolution",
    "Avoids sounding robotic or defensive",
    "Maintains a professional and calm tone",
    "Mentions urgency and prioritizes the case",
    "Does not blindly reject the request because warranty expired",
    "Keeps the response concise and readable",
]

# ── Generic prompt (weak prompt) ───────────────────────────────────────────
generic_prompt = """
    Write a customer support response email to 
    a customer who purchased a laptop 8 months ago for €1200 
    and the laptop battery has started swelling and overheating.
"""

# ── Optimized prompt (best practices applied) ──────────────────────────────
# Best practices used:
#   1. Explicit role/persona
#   2. Emotional context handling
#   3. Structured objectives
#   4. Tone constraints
#   5. Safety/brand-awareness
#   6. Anti-pattern instructions
#   7. Output length guidance

optimized_prompt = """
    You are an expert customer support representative for a premium electronics company.

    Write a professional customer support email responding to a customer who:
    - Purchased a laptop 8 months ago for €1200
    - Reports that the battery is swelling and overheating
    
    Your response should:
    1. Show empathy and acknowledge the seriousness of the issue
    2. Clearly mention that battery swelling can be a safety concern
    3. Offer a concrete next step or resolution
    4. Reassure the customer that the issue will be prioritized
    5. Maintain a calm, professional, human tone
    
    Constraints:
    - Keep the response under 180 words
    - Be concise and easy to understand
    - Do not sound robotic or overly corporate
    - Avoid unnecessary filler language
    
    Structure:
    - Short empathetic opening
    - Clear resolution/action step
    - Professional closing
"""

eval_cases = [{"prompt_text": generic_prompt, "prompt_label": "Generic Prompt"}, 
              {"prompt_text": optimized_prompt, "prompt_label": "Optimized Prompt"}]

---
## Section 3: Generate Model Responses

In [19]:
MODEL = "llama3"

# ── Helper: call the model ──────────────────────────────────────────────────
def query_model(prompt: str, system: str = "") -> str:
    """Send a prompt to Model and return the text response."""
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.append({"role": "system", "content": system})
    kwargs = {"model": MODEL, "messages": messages}
    response = ollama.chat(**kwargs)
    return response["message"]["content"]


# ── Helper: pretty-print a response ────────────────────────────────────────
def display_response(label: str, text: str, width: int = 90) -> None:
    border = "─" * width
    print(f"\n┌{border}┐")
    print(f"│ {label:<{width-1}}│")
    print(f"├{border}┤")
    for line in textwrap.wrap(text, width - 2):
        print(f"│ {line:<{width-1}}│")
    print(f"└{border}┘")

df_data = {}

---
## Section 4: Manual Evaluation (Human-in-the-Loop)

### Scoring Rubric (0.0 – 1.0 per metric)

| Score | Meaning |
|-------|---------|
| 0.0 – 0.3 | Poor – major issues |
| 0.4 – 0.6 | Acceptable – some issues |
| 0.7 – 0.8 | Good – minor issues |
| 0.9 – 1.0 | Excellent – meets all criteria |

**Accuracy**: Cross-check against the reference points above.  
**Relevance**: Does it answer the actual question without going off-topic?  
**Completeness**: How many reference points are covered?  
**Conciseness**: Is it appropriately brief, or padded/too terse?  
**Clarity**: Could a beginner genuinely understand this?  

In [22]:
METRICS = ["accuracy", "relevance", "completeness", "conciseness", "clarity"]

def overall(accuracy=0, relevance=0, completeness=0, conciseness=0, clarity=0) -> float:
    weights = {"accuracy": 0.30, "relevance": 0.25,
               "completeness": 0.20, "conciseness": 0.10, "clarity": 0.15}
    return round(
        accuracy    * weights["accuracy"]    +
        relevance   * weights["relevance"]   +
        completeness * weights["completeness"] +
        conciseness * weights["conciseness"] +
        clarity     * weights["clarity"],
        3
    )

def manual_evaluation(metrics):
    scores = {}
    for metric in metrics:
        score = float(input(f"Score for {metric} (0-1): "))
        print(f"{metric}: {score}")
        scores[metric] = score
    return scores

print("Querying model for each prompt...\n")

for case in eval_cases:
    prompt = case["prompt_text"]
    prompt_label = case['prompt_label']
    print(f"\n Evaluating {prompt_label}")
    response = query_model(prompt)
    display_response(f"{prompt_label}", response)
    scores = manual_evaluation(METRICS)
    overall_score = overall(**scores)
    print(f"Overall Score after Manual Evaluation: {overall_score}")
    scores["overall"] = overall_score
    df_data[("Human Evaluation", prompt_label)] = scores

print("\n Responses generated.")

Querying model for each prompt...


 Evaluating Generic Prompt

┌──────────────────────────────────────────────────────────────────────────────────────────┐
│ Generic Prompt                                                                           │
├──────────────────────────────────────────────────────────────────────────────────────────┤
│ Subject: Concern with Your Laptop Battery - We're Here to Help!  Dear [Customer's Name], │
│ I hope this email finds you well. I am writing to address your concern regarding your    │
│ [Brand] laptop, which you purchased from us approximately 8 months ago for €1200. We     │
│ appreciate you reaching out to us about the issue with the battery swelling and          │
│ overheating.  First and foremost, please be assured that we take the safety and          │
│ performance of our products very seriously. We understand how frustrating it must be to  │
│ experience this issue with your laptop, and I want to assure you that we're committed to │
│ reso

Score for accuracy (0-1):  0.9


accuracy: 0.9


Score for relevance (0-1):  0.8


relevance: 0.8


...

completeness: 0.9


Score for conciseness (0-1):  0.9


conciseness: 0.9


Score for clarity (0-1):  0.9


clarity: 0.9
Overall Score after Manual Evaluation: 0.9

 Responses generated.


---
## Section 5: Automated Evaluation (LLM-as-Judge)

Manual evaluation doesn't scale. When you have dozens of prompts and hundreds of test cases, you need automation.

**LLM-as-Judge** is the industry standard approach: you ask a capable language model to evaluate another model's output using a structured rubric. 

In [21]:
JUDGE_SYSTEM_PROMPT = """You are an impartial evaluation assistant. Your job is to score 
AI responses on a set of quality dimensions. You MUST respond with valid JSON only — 
no preamble, no markdown fences, no extra text."""

def build_judge_prompt(question: str, response: str, reference_points: List[str]) -> str:
    ref_list = "\n".join(f"- {rp}" for rp in reference_points)
    return f"""Evaluate the following AI response to the question below.

        QUESTION:
        {question}
        
        REFERENCE KEY POINTS (a high-quality answer should cover these):
        {ref_list}
        
        AI RESPONSE TO EVALUATE:
        {response}
        
        Score the response on each dimension from 0.0 to 1.0 based on the metrics definitions and scoring rubric:

        Metrics Definition:
        - accuracy: Are the facts stated correct and free from errors?
        - relevance: Does the response directly address the question without going off-topic?
        - completeness: How many of the reference key points are adequately covered?
        - conciseness: Is the length appropriate — not padded, not too terse?
        - clarity: Is the language clear and accessible to the intended audience?

        Scoring Rubric:
        | Score | Meaning |
        |-------|---------|
        | 0.0 – 0.3 | Poor – major issues |
        | 0.4 – 0.6 | Acceptable – some issues |
        | 0.7 – 0.8 | Good – minor issues |
        | 0.9 – 1.0 | Excellent – meets all criteria |
        
        Return ONLY a JSON object like this:
        {{
          "accuracy": <float>,
          "relevance": <float>,
          "completeness": <float>,
          "conciseness": <float>,
          "clarity": <float>
        }}"""

def automated_evaluation(prompt, response):
    judge_prompt = build_judge_prompt(
        prompt, response, REFERENCE_POINTS
    )
    raw = query_model(judge_prompt, system=JUDGE_SYSTEM_PROMPT)
    
    # Strip markdown fences if the model adds them
    raw = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    data = json.loads(raw)
    return {
        "accuracy": data["accuracy"],
        "relevance": data["relevance"],
        "completeness": data["completeness"],
        "conciseness": data["conciseness"],
        "clarity": data["clarity"],
    }

for case in eval_cases:
    prompt = case["prompt_text"]
    prompt_label = case['prompt_label']
    print(f"\n Evaluating {prompt_label}")
    response = query_model(prompt)
    display_response(f"{prompt_label}", response)
    scores = automated_evaluation(prompt, response)
    overall_score = overall(**scores)
    print(f"Overall Score after Automated Evaluation: {overall_score}")
    scores["overall"] = overall_score
    df_data[("Automated Evaluation", prompt_label)] = scores
    time.sleep(1) 

print("\n Responses generated.")


 Evaluating Generic Prompt

┌──────────────────────────────────────────────────────────────────────────────────────────┐
│ Generic Prompt                                                                           │
├──────────────────────────────────────────────────────────────────────────────────────────┤
│ Subject: Concern with Your Laptop Battery - Assistance and Solution Offered  Dear        │
│ [Customer's Name],  I am writing to express my sincerest apologies for the issues you    │
│ have been experiencing with your laptop battery, which appears to be swelling and        │
│ overheating. I understand how frustrating this must be, especially since you purchased   │
│ the device just eight months ago.  At [Company Name], we take pride in providing high-   │
│ quality products, and it's clear that we fell short of our standards in this case. Your  │
│ satisfaction is our top priority, and I want to assure you that we're committed to       │
│ resolving this issue as quickly and eff

---
## Section 6: Comparing the Results

In [24]:
df = pd.DataFrame(df_data)
# Round values for cleaner display
df = df.round(2)

# Display nicely in Jupyter Notebook
display(df)

Automated Evaluation                  Human Evaluation  \
                   Generic Prompt Optimized Prompt   Generic Prompt   
accuracy                     1.00             1.00             0.90   
relevance                    0.90             1.00             0.80   
completeness                 0.80             0.90             0.90   
conciseness                  0.70             0.80             0.70   
clarity                      0.90             0.90             0.80   
overall                      0.89             0.94             0.84   

                               
             Optimized Prompt  
accuracy                  0.9  
relevance                 0.9  
completeness              0.9  
conciseness               0.9  
clarity                   0.9  
overall                   0.9